In [1]:
import re
import csv

log_file = "app.log"
csv_file = "logs_convertidos.csv"

pattern = re.compile(
    rb'^(.*?) - ([\w\.]+) - line: (\d+) - (\w+) - (.*)$'
)

rows = []

with open(log_file, "rb") as f:   # lê em modo binário
    for line in f:
        line = line.rstrip(b"\n")

        # Tenta UTF-8, senão Latin-1, senão apela
        try:
            decoded = line.decode("utf-8")
        except UnicodeDecodeError:
            try:
                decoded = line.decode("latin-1")
            except UnicodeDecodeError:
                decoded = line.decode("latin-1", errors="replace")

        match = re.match(
            r'^(.*?) - ([\w\.]+) - line: (\d+) - (\w+) - (.*)$',
            decoded
        )

        if match:
            timestamp, filename, line_number, level, message = match.groups()
            rows.append({
                "timestamp": timestamp,
                "file": filename,
                "line": line_number,
                "level": level,
                "message": message,
            })
        else:
            rows.append({
                "timestamp": "",
                "file": "",
                "line": "",
                "level": "",
                "message": decoded
            })

# Salva CSV sem chances de crash
with open(csv_file, "w", newline="", encoding="utf-8") as csvout:
    writer = csv.DictWriter(
        csvout,
        fieldnames=["timestamp", "file", "line", "level", "message"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"CSV gerado com sucesso: {csv_file}")


CSV gerado com sucesso: logs_convertidos.csv


---

# 🚦 **1. Taxa de Erros vs. Sucesso**

Calcular:

- Quantidade de logs por `level`
- % de logs `ERROR` e `WARNING` ao longo do tempo
- Quais arquivos / módulos mais geram erro

**Insights possíveis:**

- Identificar os pontos mais frágeis do sistema
- Detectar regressões após deploys
- Medir estabilidade da app nos últimos dias/semanas

---

# ⏱ **2. Tempo de Resposta da Aplicação (Performance)**

A cada ciclo:

1. Capturar timestamp do início de uma requisição (ex: “Agent initiated”)
2. Capturar timestamp do fim (ex: “Execution completed”)
3. Calcular duração

Você pode então:

- Plotar gráfico diário de tempo médio de execução
- Detectar horários mais lentos (pico de carga)
- Descobrir se o app está piorando com o tempo

**Insights estratégicos:**

- Saber quando escalar servidor
- Melhorar UX reduzindo latência
- Identificar gargalos nas ferramentas (Pinecone, MongoDB, OpenAI…)

---

# 🧠 **3. Análise de Sessões / Uso Real da Plataforma**

Cada `session_id` pode virar uma jornada de usuário:

- Quantas mensagens por sessão?
- Sessões finalizadas vs. interrompidas
- Tempo médio de navegação/uso

Isso permite:

- Medir engajamento real
- Ver abandono em certos pontos do fluxo
- Medir impacto após nova feature

---

# 🚨 **4. “Narrativa do Erro” – Árvore de Causas**

Ao analisar erros, podemos montar algo como:

- Qual evento precede erros com mais frequência?
- Qual módulo dispara mais exceptions depois de executar outro?

Exemplo:

```
buffer_memory.py → retrieval.py → AgentAsk.py → ERROR

```

Isso revela:

- Problemas em cadeia
- Ponto exato onde o sistema degrada
- Onde otimizar para maior impacto

---

# 📈 **5. Saúde do Sistema por Módulo**

Agrupando por `file`:

- Top 10 arquivos que geram mais log
- Quais mais geram INFO ou ERROR
- Quais quase nunca aparecem → possíveis pontos sem monitoramento

Insight:

> “70% dos problemas vêm de apenas 3 módulos.”
> 

---

# 📦 **6. Telemetria de Recursos Internos**

Pelos logs, é possível analisar:

- Conexão com MongoDB
- Status do Pinecone
- Inicialização do modelo OpenAI
- Carregamento de histórico

Se em algum dia/horário ocorrer:

- Mais falhas de conexão
- Mais tempo de inicialização

Você consegue **precificar impacto real de infra**.

---

# 🔐 **7. Análise de Ativação de Clientes / Planos**

Você logou:

```
Status do plano da empresa: deactivated

```

Isso já permite:

- Saber quantas requisições vêm de contas ativas vs. inativas
- Identificar tentativa de uso após expiração
- Criar painel de:

| Métrica | Valor |
| --- | --- |
| Empresas com plano ativo | X |
| Empresas tentando usar mesmo desativadas | Y |

Pode até disparar:

- Alertas
- Emails automáticos
- Campanhas de retenção

---

# 🧰 **8. Monitorar uso de cada “Tool”**

Você tem:

```
Configurando as tools
Tools configuradas
Executando ferramenta X

```

Isso permite:

- Saber quais ferramentas são mais usadas
- Ver quais quase ninguém usa → pode remover!
- Identificar gargalos — ex: ferramenta X sempre gera demora

---

# 💾 **9. Consistência do Memory System**

Há logs como:

```
Nenhum histórico encontrado para session_id
Histórico salvo com sucesso

```

Podemos medir:

- % de session_ids sem histórico → problemas ou comportamento normal?
- Latência de salvar mensagens
- Queda ou duplicação de sessões

---

# 🧳 **10. Previsão e Detecção de Anomalias**

Com dados estruturados é possível:

- Rodar LSTM / Prophet / regressão
- Detectar picos anormais de:
    - erros
    - lentidão
    - inicializações
    - consumo de tokens
    - falhas de banco

Gerando alertas automáticos, ex.:

> “O número de erros de buffer_memory.py está 400% acima do normal nas últimas 2 horas.”
> 

---

# 👁 **11. Telemetria de Token / Custos**

Você já loga algo como:

```
Enviando dados de uso de tokens

```

Podemos:

- Calcular custo diário por cliente
- Saber quem consome mais
- Prever despesa futura
- Precificar planos de forma realista

Isso é ouro.

---

# 🧩 **12. Jornada da IA (Trail Logging)**

Você pode reconstruir o passo-a-passo de cada resposta:

1. Inicialização do agente
2. Carregamento do memory
3. Escolha de ferramenta
4. Execução de cada passo
5. Resposta final

Isso permite:

- Depuração muito mais rápida
- Explicar internamente como o LLM tomou a decisão
- Mostrar o raciocínio sem expor do OpenAI

---

# 🏁 **13. SLA de Resposta**

Para cada interação:

- Latência média
- Latência P95 / P99
- Grupo por cliente

Insight:

> “Clientes grandes têm 40% mais latência, precisamos dar prioridade a eles.”
> 

---

# 🧿 **14. Heatmap de Uso por Hora da Semana**

Agrupando por timestamp:

- Quais horários têm mais tráfego
- Quais horários geram mais erro
- Poder prever demandas futuras

---

# 📊 **15. “Ciclo de Vida de Passos Internos”**

Você pode medir:

- Quanto tempo gasta configurando Pinecone
- Quanto tempo gasta carregando memória
- Quanto tempo leva a chamada à OpenAI

Resultado:

> “Só o setup interno já consome 40% da resposta. Otimização interna traria redução de 700ms por requisição.”
> 

---

# 🎖 **Conclusão – o que dá pra descobrir**

Com esses logs você consegue:

✔ Entender **performance, estabilidade e saúde do sistema**

✔ Detectar **gargalos de IA ou infraestrutura**

✔ Identificar **o que clientes realmente usam**

✔ Reduzir custos

✔ Melhorar experiência do usuário

✔ Criar alertas automatizados

✔ Reforçar suas decisões de roadmap com **dados reais**

Se quiser, posso agora também:

- Criar dashboards em Pandas + Plotly
- Criar versão em Streamlit para monitoramento
- Escrever consultas prontas em Pandas
- Criar alertas automáticos
- Gerar análises estatísticas mais profundas